# Демонстрация 1: IMDB many-to-one классификация тональности

Перед запуском полезно открыть dataset card [IMDB](./cards/01_imdb.md).

Fast-mode идея:
- взять небольшую подвыборку;
- паддить отзывы до общей длины;
- сравнить `SimpleRNN`, `LSTM`, `GRU` на одной и той же задаче.


In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(11)
np.random.seed(11)
random.seed(11)


2026-03-24 10:59:01.654821: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 10:59:01.655225: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 10:59:01.656998: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 10:59:01.662085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774339141.670833  190960 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774339141.67

## Выбор runtime

Здесь вы выбираете, где и на чём запускать notebook.

Что обычно выбирать:
- `auto` — лучший вариант по умолчанию. Если TensorFlow видит GPU, будет выбран GPU. Если GPU нет, notebook спокойно останется на CPU.
- `local-cpu` — локальный запуск только на CPU, даже если видеокарта есть.
- `local-gpu` — локальный запуск с обязательным GPU. Если GPU не настроен, notebook специально остановится с понятной ошибкой.
- `colab-cpu` / `colab-gpu` — запуск в Google Colab.
- `kaggle-cpu` / `kaggle-gpu` — запуск в Kaggle Notebooks.

Что важно:
- после изменения `RUNTIME_MODE` используйте `Restart & Run All`;
- `COURSE_REPO_HTTPS_URL` нужен только для Colab/Kaggle, если репозиторий ещё не клонирован в runtime;
- пока в ячейке стоит placeholder-URL, cloud auto-bootstrap не сможет сам скачать курс;
- guide `05` отвечает на вопрос, где и как запускать notebook;
- guide `06` нужен, если вы хотите именно локальный GPU и не уверены в версиях `TensorFlow` / `CUDA`;
- локальный GPU-path курса: `Linux + NVIDIA` или `Windows -> WSL2 + Ubuntu`;
- если `local-gpu` упирается в локальные CUDA/PTX ошибки, это обычно уже проблема GPU-стека, а не notebook. В таком случае спокойно переключайтесь на `local-cpu`, `colab-gpu` или `kaggle-gpu`.

Подробные guides:
- `themes/00-Foundations/guides/05_local_tensorflow_gpu_notebooks.md`
- `themes/00-Foundations/guides/06_tensorflow_cuda_version_selection.md`


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/00-Foundations/requirements.txt"


def _detect_notebook_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


In [2]:
vocab_size = 8000
maxlen = 140
train_subset = 3000
test_subset = 1500

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

x_train = tf.keras.utils.pad_sequences(x_train[:train_subset], maxlen=maxlen, padding='post')
y_train = y_train[:train_subset]
x_test = tf.keras.utils.pad_sequences(x_test[:test_subset], maxlen=maxlen, padding='post')
y_test = y_test[:test_subset]

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=11, stratify=y_train
)

print('x_train shape:', x_train.shape)
print('x_val shape  :', x_val.shape)
print('x_test shape :', x_test.shape)
print('Positive rate:', y_train.mean())


       0/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   16384/17464789 ━━━━━━━━━━━━━━━━━━━━ 1:44 6us/step

   57344/17464789 ━━━━━━━━━━━━━━━━━━━━ 1:00 4us/step

  114688/17464789 ━━━━━━━━━━━━━━━━━━━━ 45s 3us/step 

  147456/17464789 ━━━━━━━━━━━━━━━━━━━━ 47s 3us/step

...

17137664/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

17334272/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

17416192/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 15s 1us/step


x_train shape: (2400, 140)
x_val shape  : (600, 140)
x_test shape : (1500, 140)
Positive rate: 0.5141666666666667


In [3]:
def build_model(cell_name: str) -> tf.keras.Model:
    if cell_name == 'SimpleRNN':
        recurrent = tf.keras.layers.SimpleRNN(32)
    elif cell_name == 'LSTM':
        recurrent = tf.keras.layers.LSTM(32)
    elif cell_name == 'GRU':
        recurrent = tf.keras.layers.GRU(32)
    else:
        raise ValueError(cell_name)

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(maxlen,)),
            tf.keras.layers.Embedding(vocab_size, 32, mask_zero=True),
            recurrent,
            tf.keras.layers.Dense(1, activation='sigmoid'),
        ]
    )
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


histories = {}
summaries = []
models = {}

for name in ['SimpleRNN', 'LSTM', 'GRU']:
    model = build_model(name)
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=2,
        batch_size=64,
        verbose=0,
    )
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    histories[name] = history.history
    summaries.append({'model': name, 'test_loss': float(loss), 'test_accuracy': float(accuracy)})
    models[name] = model

pd.DataFrame(summaries).sort_values('test_accuracy', ascending=False)


E0000 00:00:1774339162.813765  190960 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774339162.816230  190960 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


2026-03-24 10:59:23.669610: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


,model,test_loss,test_accuracy
1,LSTM,0.576488,0.708000
2,GRU,0.612001,0.698000
0,SimpleRNN,0.696306,0.537333


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history['val_accuracy'], marker='o', label=name)
    axes[1].plot(history['val_loss'], marker='o', label=name)

axes[0].set_title('IMDB fast-mode: validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].set_title('IMDB fast-mode: validation loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


/tmp/ipykernel_190960/787849481.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
sample_batch = x_test[:4]
for name, model in models.items():
    probs = model.predict(sample_batch, verbose=0).ravel()
    preds = (probs >= 0.5).astype(np.int32)
    print(name)
    for idx, (prob, pred, target) in enumerate(zip(probs, preds, y_test[:4]), start=1):
        print(f'  sample={idx} prob={prob:.3f} pred={pred} target={int(target)}')
    print()


SimpleRNN
  sample=1 prob=0.321 pred=0 target=0
  sample=2 prob=0.603 pred=1 target=1
  sample=3 prob=0.671 pred=1 target=1
  sample=4 prob=0.407 pred=0 target=0

LSTM
  sample=1 prob=0.404 pred=0 target=0
  sample=2 prob=0.435 pred=0 target=1
  sample=3 prob=0.391 pred=0 target=1
  sample=4 prob=0.473 pred=0 target=0



GRU
  sample=1 prob=0.495 pred=0 target=0
  sample=2 prob=0.509 pred=1 target=1
  sample=3 prob=0.486 pred=0 target=1
  sample=4 prob=0.547 pred=1 target=0



## Что заметить

- Логика задачи осталась той же, что и в `many-to-one` toy notebook: одна последовательность -> одна метка.
- Из synthetic labs сюда почти без изменений переезжает shape `(batch, time)` для token ids и итоговая метка `(batch,)`.
- Разница между `SimpleRNN`, `LSTM` и `GRU` видна даже в fast-mode, но этот notebook нужен прежде всего для переноса интуиции, а не для честного leaderboard.
